# LLM Adaptation Workflow
### Notebook 06 — Evaluation & Benchmarking

---

## Why evaluation matters

Every step in this workflow — fine-tuning, RAG, alignment — makes claims about improving the model. Without rigorous evaluation, those claims are just guesses.

**Evaluation answers the questions that matter to a business:**
- Is the adapted model actually better at our tasks?
- Does it hallucinate less?
- Is the improvement worth the cost and complexity?
- Where does it still fail?

This notebook builds a complete evaluation framework covering:
1. **Automated text quality metrics** (ROUGE, BERTScore)
2. **Hallucination detection**
3. **Before vs after comparison** across all adaptation steps
4. **Cost/performance trade-off analysis**

---

## The metrics

| Metric | What it measures | How it works |
|--------|-----------------|-------------|
| **ROUGE-L** | Text overlap | Longest common subsequence between generated and reference text |
| **BERTScore** | Semantic similarity | Embeddings-based similarity — catches paraphrases that ROUGE misses |
| **Factual consistency** | Hallucination | NLI model checks if response is entailed by the source context |
| **Exact answer match** | Factual accuracy | Does the response contain the correct fact/number? |

No single metric tells the full story. We use all four together.

In [ ]:
import torch
import json
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from peft import PeftModel
import evaluate
from bert_score import score as bert_score_fn

warnings.filterwarnings("ignore")

MODEL_DIR = Path("../models")

device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Using device: {device}")

---

## 1. Evaluation benchmark

We define a set of test questions with reference answers. These are held out — never used in training — so they give an unbiased measure of performance.

**Important**: good evaluation questions should:
- Cover the range of tasks the model will face in production
- Have clear, verifiable ground-truth answers
- Include both easy and challenging cases
- Test for hallucination (questions the model might confabulate)

In [ ]:
# Our evaluation benchmark — 10 questions spanning different difficulty levels
# These were NOT used in fine-tuning (held out for evaluation only)

benchmark = [
    {
        "question": "What was Apple's total net sales for fiscal year 2023?",
        "reference": "Apple's total net sales for fiscal year 2023 were $383.3 billion, down 2.8% from $394.3 billion in fiscal 2022.",
        "key_fact": "383.3 billion",
        "context": "Apple reported total net sales of $383.3 billion for the fiscal year ended September 30, 2023.",
        "difficulty": "easy"
    },
    {
        "question": "By how much did Microsoft's Azure grow in fiscal year 2023?",
        "reference": "Microsoft Azure and cloud services revenue grew 27% in fiscal year 2023.",
        "key_fact": "27%",
        "context": "Azure and other cloud services revenue grew 27% in fiscal 2023.",
        "difficulty": "easy"
    },
    {
        "question": "What percentage of Apple's revenue comes from international markets?",
        "reference": "International markets accounted for 58% of Apple's total revenues.",
        "key_fact": "58%",
        "context": "International markets accounted for 58% of total Apple revenues in fiscal 2023.",
        "difficulty": "easy"
    },
    {
        "question": "What was JPMorgan Chase's net income in 2023 and what drove it?",
        "reference": "JPMorgan Chase reported net income of $49.6 billion for fiscal year 2023, driven primarily by higher net interest income as interest rates rose throughout the year.",
        "key_fact": "49.6 billion",
        "context": "JPMorgan Chase reported net income of $49.6 billion for fiscal year 2023, driven by higher net interest income as interest rates rose.",
        "difficulty": "medium"
    },
    {
        "question": "Explain the relationship between Apple's Products and Services segments.",
        "reference": "Apple operates two main revenue segments: Products ($298.1B) covering hardware like iPhone, Mac, and iPad, and Services ($85.2B) covering App Store, Apple Music, iCloud, and Apple TV+. Services grew 16% while Products declined, reflecting Apple's ongoing strategic shift toward recurring software revenue.",
        "key_fact": "Services grew 16",
        "context": "Apple's Products revenue was $298.1 billion and Services revenue was $85.2 billion, growing 16% year-over-year.",
        "difficulty": "medium"
    },
    {
        "question": "What were Microsoft's three business segments and how did each perform?",
        "reference": "Microsoft operates three segments: Productivity and Business Processes (Office, LinkedIn), Intelligent Cloud (Azure, server products), and More Personal Computing (Windows, Xbox). Intelligent Cloud was the standout performer, growing 19% driven by Azure's 27% growth.",
        "key_fact": "Intelligent Cloud",
        "context": "Microsoft operates through Productivity and Business Processes, Intelligent Cloud, and More Personal Computing segments. Intelligent Cloud generated $87.9B, up 19%.",
        "difficulty": "medium"
    },
    {
        "question": "How does JPMorgan Chase approach credit risk management?",
        "reference": "JPMorgan manages credit risk through portfolio diversification, counterparty credit limits, collateral requirements to secure exposures, and hedging with credit derivatives. This is supplemented by internal credit rating models and regular stress testing of the portfolio.",
        "key_fact": "diversification",
        "context": "JPMorgan manages credit risk through diversification, credit limits, collateral requirements, and credit derivatives, supplemented by internal ratings and stress testing.",
        "difficulty": "hard"
    },
    {
        "question": "What are the investment risks unique to technology companies compared to financial companies?",
        "reference": "Technology companies face rapid obsolescence risk, intense R&D investment requirements, and concentration in a few key products. Financial companies face interest rate sensitivity, credit cycle risk, and regulatory capital requirements. Both face cybersecurity risk, but for different reasons — technology companies are high-value targets for data theft, while financial companies face direct financial loss from fraud.",
        "key_fact": None,  # open-ended question
        "context": None,
        "difficulty": "hard"
    },
]

print(f"Benchmark: {len(benchmark)} questions")
print(f"  Easy  : {sum(1 for q in benchmark if q['difficulty'] == 'easy')}")
print(f"  Medium: {sum(1 for q in benchmark if q['difficulty'] == 'medium')}")
print(f"  Hard  : {sum(1 for q in benchmark if q['difficulty'] == 'hard')}")

---

## 2. Load models for comparison

In [ ]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

base_tokeniser = AutoTokenizer.from_pretrained(MODEL_ID)
if base_tokeniser.pad_token is None:
    base_tokeniser.pad_token = base_tokeniser.eos_token

# 1. Base model (no adaptation)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)

# 2. Fine-tuned model (from Notebook 04)
adapter_path = MODEL_DIR / "tinyllama-financial-lora" / "final_adapter"
if adapter_path.exists():
    ft_base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, device_map="auto"
    )
    ft_model = PeftModel.from_pretrained(ft_base, str(adapter_path))
    models = {"base": base_model, "fine_tuned": ft_model}
    print("Loaded: base model + fine-tuned model")
else:
    models = {"base": base_model}
    print("Loaded: base model only (run Notebook 04 to add fine-tuned model)")

In [ ]:
def generate(model, tokeniser, question, context=None, max_new_tokens=150):
    system = "You are a financial analyst assistant. Answer accurately and concisely."
    user = f"{question}" if context is None else f"Context: {context}\n\nQuestion: {question}"
    
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ]
    formatted = tokeniser.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokeniser(formatted, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,   # Low temperature for more deterministic evaluation
            do_sample=True,
            pad_token_id=tokeniser.eos_token_id,
        )
    return tokeniser.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


# Generate responses from all models for all benchmark questions
print("Generating responses for all models and benchmark questions...")
results = {name: [] for name in models}

for q in benchmark:
    for model_name, model in models.items():
        response = generate(model, base_tokeniser, q["question"], q.get("context"))
        results[model_name].append(response)

print("Done.")

---

## 3. ROUGE and BERTScore

In [ ]:
# ROUGE: measures n-gram overlap between generated text and reference answers
# Good for measuring if key facts/phrases appear in the response

rouge = evaluate.load("rouge")
references = [q["reference"] for q in benchmark]

print("ROUGE-L scores (higher = better text overlap with reference answer):")
print()
rouge_scores = {}
for model_name, preds in results.items():
    scores = rouge.compute(predictions=preds, references=references)
    rouge_scores[model_name] = scores["rougeL"]
    print(f"  {model_name:15s}: ROUGE-L = {scores['rougeL']:.3f}")

print()
print("ROUGE-L interpretation:")
print("  < 0.20: poor overlap (likely missing key facts or different phrasing)")
print("  0.20-0.35: moderate overlap")
print("  > 0.35: good overlap (responses capture key information from reference)")

In [ ]:
# BERTScore: measures semantic similarity using embeddings
# Better than ROUGE because it catches paraphrases and synonyms
# e.g. "revenue increased" and "sales grew" score high — ROUGE would miss this

print("BERTScore F1 (higher = better semantic similarity to reference):")
print()
bert_scores = {}
for model_name, preds in results.items():
    _, _, F1 = bert_score_fn(
        preds,
        references,
        lang="en",
        verbose=False,
        device=device if device != "mps" else "cpu"  # BERTScore runs on CPU if MPS unavailable
    )
    mean_f1 = F1.mean().item()
    bert_scores[model_name] = mean_f1
    print(f"  {model_name:15s}: BERTScore F1 = {mean_f1:.3f}")

print()
print("BERTScore F1 interpretation:")
print("  < 0.85: responses differ substantially in meaning from reference")
print("  0.85-0.90: reasonable semantic alignment")
print("  > 0.90: high semantic similarity to reference")

---

## 4. Hallucination detection

Hallucination — generating plausible but incorrect facts — is one of the most serious problems in production LLM systems, especially in financial contexts where accuracy matters.

We use **Natural Language Inference (NLI)**: a model trained to determine if one piece of text *entails* or *contradicts* another. If a model's response contradicts the source context, that's a hallucination signal.

In [ ]:
# Load an NLI model for factual consistency checking
# facebook/bart-large-mnli: trained on Multi-NLI, robust for factual consistency

nli_pipeline = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if device == "cuda" else -1  # -1 = CPU, 0 = GPU
)

print("NLI model loaded.")

In [ ]:
def check_factual_consistency(context: str, claim: str) -> float:
    """
    Check if a claim is consistent with (entailed by) a context.
    Returns a score between 0 and 1:
      - Close to 1 = consistent (no hallucination)
      - Close to 0 = likely contradicts the context (potential hallucination)
    """
    # We ask: does this context ENTAIL the claim?
    result = nli_pipeline(
        sequences=context[:512],     # NLI models have token limits
        candidate_labels=["entailment", "contradiction", "neutral"],
        hypothesis_template="The following is true: {}",
        multi_label=False,
    )
    
    # Return the entailment probability — how well the context supports the claim
    label_scores = dict(zip(result["labels"], result["scores"]))
    return label_scores.get("entailment", 0.0)


# Test hallucination detection on examples with known contexts
test_cases = [
    {
        "context": "Apple reported total net sales of $383.3 billion for fiscal year 2023.",
        "claim_correct": "Apple's total net sales were $383.3 billion in 2023.",
        "claim_hallucinated": "Apple's total net sales were $450 billion in 2023.",
    },
    {
        "context": "Microsoft Azure revenue grew 27% in fiscal 2023.",
        "claim_correct": "Azure cloud services grew approximately 27% year over year.",
        "claim_hallucinated": "Microsoft Azure revenue declined in fiscal 2023.",
    }
]

print("Hallucination detection test:")
print(f"{'':45s} {'Entailment':>12}")
print("-" * 60)
for tc in test_cases:
    score_correct     = check_factual_consistency(tc["context"], tc["claim_correct"])
    score_hallucinated = check_factual_consistency(tc["context"], tc["claim_hallucinated"])
    print(f"  [Correct]     {tc['claim_correct'][:45]:45s} {score_correct:>10.3f}")
    print(f"  [Hallucinated]{tc['claim_hallucinated'][:45]:45s} {score_hallucinated:>10.3f}")
    print()

In [ ]:
# Now apply hallucination detection to model responses
# We check responses to questions that have a known context (not open-ended questions)

print("Hallucination rate by model (fraction of responses that score < 0.3 on entailment):")
print()

hallucination_rates = {}
for model_name, preds in results.items():
    hallucinated_count = 0
    total_with_context = 0
    
    for i, (pred, q) in enumerate(zip(preds, benchmark)):
        if q["context"] is not None:  # Only check questions with known context
            score = check_factual_consistency(q["context"], pred[:300])
            if score < 0.3:
                hallucinated_count += 1
            total_with_context += 1
    
    rate = hallucinated_count / total_with_context if total_with_context > 0 else 0
    hallucination_rates[model_name] = rate
    print(f"  {model_name:15s}: {rate:.0%} hallucination rate  ({hallucinated_count}/{total_with_context} responses)")

print()
print("Note: lower is better. A 0% hallucination rate means all responses")
print("were factually consistent with the provided context.")

---

## 5. Key fact accuracy

In [ ]:
# For questions with a specific expected fact (number, percentage, name),
# we check if that fact appears in the response.
# This is a simple but highly informative metric for factual Q&A.

print("Key fact accuracy (does the response contain the key fact?):")
print()

factual_questions = [q for q in benchmark if q["key_fact"] is not None]

for model_name, preds in results.items():
    relevant_preds = [preds[i] for i, q in enumerate(benchmark) if q["key_fact"] is not None]
    hits = sum(
        1 for pred, q in zip(relevant_preds, factual_questions)
        if q["key_fact"].lower() in pred.lower()
    )
    acc = hits / len(factual_questions)
    print(f"  {model_name:15s}: {acc:.0%}  ({hits}/{len(factual_questions)} facts correct)")

print()
print("Breakdown by question:")
for i, q in enumerate(factual_questions):
    print(f"  Q: {q['question'][:60]:60s}  key: '{q['key_fact']}'")
    for model_name, preds in results.items():
        relevant_idx = [j for j, bq in enumerate(benchmark) if bq["key_fact"] is not None][i]
        found = q["key_fact"].lower() in preds[relevant_idx].lower()
        print(f"    {model_name:15s}: {'✓' if found else '✗'}")

---

## 6. Summary dashboard

In [ ]:
# Compile all metrics into a summary dataframe
summary_data = []
for model_name in results:
    row = {
        "Model": model_name,
        "ROUGE-L": rouge_scores.get(model_name, 0),
        "BERTScore F1": bert_scores.get(model_name, 0),
        "Hallucination rate": hallucination_rates.get(model_name, 0),
    }
    summary_data.append(row)

summary_df = pd.DataFrame(summary_data).set_index("Model")

print("=" * 60)
print("EVALUATION SUMMARY")
print("=" * 60)
print()
print(summary_df.to_string(float_format="{:.3f}".format))
print()
print("Note: Higher ROUGE-L and BERTScore are better.")
print("      Lower Hallucination Rate is better.")

In [ ]:
# Plot the results
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Model Evaluation: Base vs Fine-tuned", fontsize=14, fontweight="bold")

model_names = list(results.keys())
colors = ["#4C72B0", "#55A868", "#C44E52"]

# ROUGE-L
axes[0].bar(model_names, [rouge_scores.get(m, 0) for m in model_names], color=colors[:len(model_names)])
axes[0].set_title("ROUGE-L (higher = better)")
axes[0].set_ylim(0, 0.6)
axes[0].axhline(y=0.3, color="gray", linestyle="--", alpha=0.5, label="good threshold")

# BERTScore
axes[1].bar(model_names, [bert_scores.get(m, 0) for m in model_names], color=colors[:len(model_names)])
axes[1].set_title("BERTScore F1 (higher = better)")
axes[1].set_ylim(0.7, 1.0)

# Hallucination rate
axes[2].bar(model_names, [hallucination_rates.get(m, 0) for m in model_names], color=colors[:len(model_names)])
axes[2].set_title("Hallucination Rate (lower = better)")
axes[2].set_ylim(0, 1.0)
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))

for ax in axes:
    ax.set_xticklabels(model_names, rotation=15, ha="right")

plt.tight_layout()
plt.savefig("../data/processed/evaluation_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved.")

---

## 7. Cost and performance trade-off

In a real deployment decision, raw performance is only one dimension. The question is always: **is the improvement worth the cost?**

In [ ]:
# Rough cost/performance analysis
# These are illustrative estimates — real costs depend heavily on scale

approaches = pd.DataFrame([
    {
        "Approach": "Base model (no adaptation)",
        "Setup time": "0 hours",
        "One-time cost": "$0",
        "Inference latency": "~150ms",
        "Knowledge update": "Retrain",
        "Domain accuracy": "Low",
        "Best for": "Prototyping",
    },
    {
        "Approach": "RAG only",
        "Setup time": "2-4 hours",
        "One-time cost": "~$50-200 (embedding compute)",
        "Inference latency": "~300-500ms (retrieval + generation)",
        "Knowledge update": "Immediate (just update the index)",
        "Domain accuracy": "Medium",
        "Best for": "Frequently changing knowledge",
    },
    {
        "Approach": "Fine-tuning (LoRA)",
        "Setup time": "4-8 hours",
        "One-time cost": "~$100-500 (training compute)",
        "Inference latency": "~150ms (same as base)",
        "Knowledge update": "Periodic retraining needed",
        "Domain accuracy": "High",
        "Best for": "Stable domain, consistent style needs",
    },
    {
        "Approach": "Fine-tuning + RAG + DPO",
        "Setup time": "2-3 days",
        "One-time cost": "~$500-2000",
        "Inference latency": "~300-500ms",
        "Knowledge update": "Update index immediately, retrain quarterly",
        "Domain accuracy": "Highest",
        "Best for": "Production enterprise deployment",
    },
])

print("Cost & Performance Trade-off")
print("=" * 90)
print(approaches.to_string(index=False))
print()
print("The 'right' choice depends on query volume, accuracy requirements, and update frequency.")
print("For most enterprise pilots: start with RAG, add fine-tuning once the use case is validated.")

---

## Summary

In this notebook we:
- Built an evaluation benchmark with held-out test questions spanning easy to hard
- Measured text quality with ROUGE-L (n-gram overlap) and BERTScore (semantic similarity)
- Detected hallucinations using an NLI model to check factual consistency against source context
- Measured key fact accuracy — whether specific numbers and facts appear correctly
- Compared all models across all metrics in a summary dashboard
- Analysed the cost, latency, and complexity trade-offs between adaptation approaches

---

## Full workflow complete

```
✓  01 · Foundation Models    — loaded and inspected an open-source LLM
✓  02 · Data Preparation     — built an instruction dataset from public filings
✓  03 · RAG Pipeline         — built a retrieval system for grounded answers
✓  04 · Fine-Tuning          — adapted model weights using LoRA
✓  05 · Alignment            — applied DPO preference optimisation
✓  06 · Evaluation           — measured quality with ROUGE, BERTScore, hallucination detection
```

To adapt this workflow to a new domain:
1. Replace the SEC filing loader in **Notebook 02** with your corpus
2. Update the system prompt in `utils/templates.py`
3. Update the benchmark questions in **Notebook 06** with domain-relevant test cases
4. Everything else runs unchanged